# Lost in Transcription (Spanish–English): fine-tune Whisper large-v3 with LoRA on Kaggle

What this notebook does:

1. Installs the package versions the competition runtime uses (transformers 4.57.6, peft 0.20.0,
   faster-whisper 1.2.1, ctranslate2 4.8.2).
2. Finds the competition data: the Bangor Miami corpus and the official dev set.
3. Turns Miami's CHAT transcripts into training clips of up to 28 s in the dev transcript style,
   and splits the dev set into two halves.
4. Fine-tunes Whisper large-v3 with LoRA on Miami plus dev half A. The checkpoint with the lowest
   WER on held-out Miami conversations is kept. Miami clips get a random Opus round trip so they
   sound more like phone voice notes.
5. Exports the model to CTranslate2. Scores it and zero-shot large-v3 on dev half B with the
   official scorer, using the submission's own decoding settings.
6. Writes `submission.zip` (`main.py` + `model/`) to the output, ready to upload.

**Setup**

- *Settings → Accelerator*: **GPU T4 ×2** (P100 also works, more slowly). *Internet*: **on**.
- **Data**: download the Miami and dev archives from the competition's data page and add them to
  this notebook as a **private** Kaggle dataset (*Add Input*). Tarballs or extracted folders both
  work. Alternatively, fill in `MDC_DATASET_IDS` below and store your Mozilla Data Collective API
  key as a Kaggle secret named `MDC_API_KEY` (*Add-ons → Secrets*).
- **Time**: about 1.5–2 h on T4 ×2 with the defaults.

**For the final model**, set `DEV_FOR_TRAINING = "all"` and run again. The submitted model then
has seen every test-like dev clip, but there is no dev score left to check it against.


In [ ]:
import os
from pathlib import Path

SMOKE = os.environ.get("LIT_SMOKE") == "1"  # tiny CPU run used to test this notebook

# ---- where things live (the environment variables are only for testing outside Kaggle)
INPUT_ROOT = Path(os.environ.get("LIT_INPUT_ROOT", "/kaggle/input"))  # attached datasets
WORK = Path(os.environ.get("LIT_WORK", "/kaggle/working"))           # saved output (20 GB limit)
SCRATCH = Path(os.environ.get("LIT_SCRATCH", "/tmp/lit"))            # big intermediate files
MDC_DATASET_IDS = {"miami": "", "enspa_dev": ""}  # only to download with the MDC API instead

# ---- experiment
BASE_MODEL = "openai/whisper-large-v3"
ZERO_SHOT_MODEL = "large-v3"       # faster-whisper name of the same model, for the baseline
DEV_FOR_TRAINING = "half"          # "none": score on all of dev | "half": train on A, score on B
                                   # "all": final model, no dev score
RUN_ZERO_SHOT_BASELINE = True      # score untuned large-v3 on the same dev clips (~5 min)
MAX_STEPS = 300                    # another team saw gains flatten after 100-200 steps
EVAL_STEPS = 50
EVAL_SAMPLES = 160                 # held-out Miami clips decoded at each evaluation
LR = 1e-4
LORA_R, LORA_ALPHA = 64, 64
ENCODER_FROM_LAYER = 16            # LoRA on encoder layers 16-31 plus the whole decoder
BATCH_PER_GPU = 8                  # fits a 16 GB T4 in fp16 with gradient checkpointing
EFFECTIVE_BATCH = 48
AUGMENT_PROB = 0.5                 # share of Miami clips passed through an Opus round trip
CT2_QUANTIZATION = "float16"       # "int8_float16" halves the zip if size becomes a problem

if SMOKE:
    BASE_MODEL, ZERO_SHOT_MODEL = "openai/whisper-tiny", "tiny"
    MAX_STEPS, EVAL_STEPS, EVAL_SAMPLES = 4, 2, 4
    BATCH_PER_GPU, EFFECTIVE_BATCH, ENCODER_FROM_LAYER = 2, 4, 2

for d in (WORK, SCRATCH):
    d.mkdir(parents=True, exist_ok=True)


In [ ]:
import importlib.util
import subprocess
import sys

if not SMOKE:
    !pip install -q "transformers==4.57.6" "peft==0.20.0" "ctranslate2==4.8.2" "faster-whisper==1.2.1" jiwer typer wordfreq rapidfuzz soundfile

# CTranslate2 opens CUDA 12 cuBLAS/cuDNN by name at the first GPU computation. Put the pip-installed
# copies on the library path so the evaluation subprocesses find them.
lib_dirs = []
for pkg in ("nvidia.cublas", "nvidia.cudnn", "nvidia.cuda_runtime"):
    try:
        spec = importlib.util.find_spec(pkg)
    except ModuleNotFoundError:
        spec = None
    for p in (spec.submodule_search_locations or []) if spec else []:
        if (Path(p) / "lib").is_dir():
            lib_dirs.append(str(Path(p) / "lib"))
os.environ["LD_LIBRARY_PATH"] = ":".join(lib_dirs + [os.environ.get("LD_LIBRARY_PATH", "")]).strip(":")

# Query GPUs with nvidia-smi so this process never opens a CUDA context (it would sit on GPU 0
# during training).
try:
    gpus = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip().splitlines()
except FileNotFoundError:
    gpus = []
N_GPUS = len(gpus)
import transformers, peft, faster_whisper, ctranslate2
print(f"python {sys.version.split()[0]} | transformers {transformers.__version__} | peft {peft.__version__} | "
      f"faster-whisper {faster_whisper.__version__} | ctranslate2 {ctranslate2.__version__}")
print(f"{N_GPUS} GPU(s): {gpus or 'none - CPU only'}")


## Code
The scripts below are written to files so `torchrun` can launch them. Their sources are collapsed; expand a cell to read or edit it.

In [ ]:
%%writefile prepare_data.py
#!/usr/bin/env python3
"""Turn the competition downloads into training / evaluation manifests with 16 kHz FLAC clips.

Inputs (as unpacked from Mozilla Data Collective):

* Bangor Miami: ``MIAMI/chat/*.cha`` CHAT transcripts and ``MIAMI/audios/<name>.mp3`` recordings.
* Official dev set: ``DEV/clips/*`` plus ``DEV/metadata.tsv`` (``audio_filename``, ``transcript``,
  optionally ``speaker``).

Miami utterances are cleaned into verbatim text in the dev transcript style (CHAT markup removed,
fillers and repetitions kept, cut-off words as ``wor...``, the dev spelling conventions applied),
merged into single-speaker clips of up to ~28 s, and split by conversation into train / holdout.
Clips with unintelligible speech or much crosstalk are dropped. Each clip gets a language label
(the majority of its words' @s tags), used as Whisper's language token in training.

Outputs in OUT/: ``clips/*.flac``, ``miami_train.csv``, ``miami_holdout.csv``, ``dev_all.csv``,
``dev_a.csv`` / ``dev_b.csv`` (a fixed split so part of dev can be trained on while the rest is
held out), ``gold_dev_all.csv`` / ``gold_dev_b.csv`` (for compare.py) and ``stats.json``.

    python prepare_data.py --miami MIAMI --dev DEV --out OUT
"""

from __future__ import annotations

import argparse
import json
import random
import re
from collections import Counter, defaultdict
from concurrent.futures import ProcessPoolExecutor
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import soundfile as sf

SR = 16000
BULLET = re.compile(r"[\x15•](\d+)_(\d+)[\x15•]")
PRECODE = re.compile(r"\[-\s*([a-z]{3})\]")
FILLERS = {"uh", "um", "eh", "ah", "er", "erm", "mm", "mhm", "hm", "hmm", "uhm", "em", "ehm", "uh-huh", "huh"}
TERMINATORS = {
    ".": ".", "?": "?", "!": "!", "+...": "...", "+..?": "?", "+!?": "?", "+/.": "—", "+//.": "—",
    "+/?": "?", "+//?": "?", "+\"/.": ".", "+\".": ".", "+.": ".", "+=.": ".", "+...?": "?",
}
LINKERS = {"+<", "+^", "+,", "++", "+\"", "+"}
STRIP_CHARS = re.compile(r"[ˈˌ↑↓≈≋∬⌈⌉⌊⌋‹›“”\"^~:]")
# Spelling differences between Miami's CHAT transcripts and the dev/test references, as reported by
# another team's audit (dev writes "going to", never "gonna"; "uh", never "ah"). Check them with
# the audit printed by this script before relying on them.
DEFAULT_CONVENTIONS = {
    "expand": {"gonna": "going to", "wanna": "want to", "gotta": "got to", "kinda": "kind of",
               "lemme": "let me", "gimme": "give me"},
    "rename": {"ah": "uh", "ok": "okay", "o_k": "okay"},
}
AUDIT_PAIRS = [("gonna", "going"), ("wanna", "want"), ("ah", "uh"), ("ok", "okay"), ("um", "uh"),
               ("yeah", "yes"), ("cause", "because"), ("pa'", "para")]


# --------------------------------------------------------------------------- CHAT parsing

@dataclass
class Utterance:
    speaker: str
    start: float  # seconds
    end: float
    text: str
    n_es: int = 0
    n_en: int = 0
    unintelligible: bool = False
    raw: str = field(default="", repr=False)


def _word_language(tag: Optional[str], default: str) -> Optional[str]:
    """Language of one word from its @s tag (None = ambiguous / both languages)."""
    if tag is None:
        return default
    if tag == "":  # bare @s: the other language of the pair
        return "en" if default == "es" else "es"
    if "&" in tag or "+" in tag:
        return None
    return {"spa": "es", "eng": "en"}.get(tag[:3], None)


def style(tokens: list[str]) -> str:
    """Join tokens in the dev style: terminators attached, sentence-initial capitals."""
    out = ""
    for tok in tokens:
        if tok in (".", "?", "!", "...", "—"):
            out = out.rstrip() + tok + " "
        else:
            out += tok + " "
    out = re.sub(r"\s+", " ", out).strip()
    # capitalise after a sentence end, but not after an ellipsis ("wor... and" stays lowercase)
    out = re.sub(r"(^|(?<!\.\.)[.?!]\s+)([a-záéíóúñü])", lambda m: m.group(1) + m.group(2).upper(), out)
    return out if re.search(r"[^\W\d_]", out) else ""


def clean_chat_line(body: str, default_lang: str, conventions: Optional[dict] = None) -> tuple[str, int, int, bool]:
    """CHAT main-tier text -> (verbatim text, n Spanish words, n English words, unintelligible)."""
    conv = conventions if conventions is not None else DEFAULT_CONVENTIONS
    s = BULLET.sub(" ", body)
    m = PRECODE.search(s)
    lang = {"spa": "es", "eng": "en"}.get(m.group(1), default_lang) if m else default_lang
    s = PRECODE.sub(" ", s)
    unintelligible = bool(re.search(r"(?<![\w&])(xxx|yyy|www)(?!\w)", s))
    s = re.sub(r"\[[^\]]*\]", " ", s)  # retracings, overlaps, explanations, replacements, error codes
    s = s.replace("<", " ").replace(">", " ")
    s = re.sub(r"\(\.+\)", " ", s)  # pauses
    tokens, n_es, n_en = [], 0, 0
    for tok in s.split():
        if tok in TERMINATORS:
            tokens.append(TERMINATORS[tok])
            continue
        if tok in LINKERS or tok.startswith("+"):
            continue
        if tok.startswith("&"):
            body_ = tok.lstrip("&")
            if body_[:1] in ("=", "~", "*"):
                continue  # events, nonwords, interposed words
            word = body_.lstrip("-+").split("@")[0]
            word = STRIP_CHARS.sub("", re.sub(r"[()]", "", word)).lower()
            if not word:
                continue
            tokens.append(conv["rename"].get(word, word) if word in FILLERS else word + "...")
            continue
        if tok in ("xxx", "yyy", "www") or tok.startswith("0"):
            continue
        tag = None
        if "@" in tok:
            tok, _, marker = tok.partition("@")
            if marker.startswith("s"):
                tag = marker[2:] if marker.startswith("s:") else ""
        word = STRIP_CHARS.sub("", tok.replace("(", "").replace(")", ""))
        word = word.replace("_", " ").replace("+", " ").strip()
        if not word or not re.search(r"[^\W\d_]", word):
            continue
        low = word.lower()
        if low in conv["expand"]:
            word = conv["expand"][low]
        elif low in conv["rename"]:
            word = conv["rename"][low]
        elif low == "i":
            word = "I"
        if word.lower() not in FILLERS:  # fillers carry no language
            wl = _word_language(tag, lang)
            n_es += wl == "es"
            n_en += wl == "en"
        tokens.append(word)
    return style(tokens), n_es, n_en, unintelligible


def parse_chat(path: Path, conventions: Optional[dict] = None) -> tuple[list[Utterance], str]:
    """Timed utterances of one .cha file (continuation lines joined, dependent tiers skipped)."""
    lines = Path(path).read_text(encoding="utf-8", errors="replace").splitlines()
    default = "en"
    for line in lines:
        if line.startswith("@Languages:"):
            first = re.split(r"[,\s]+", line.split(":", 1)[1].strip())[0]
            default = {"spa": "es", "eng": "en"}.get(first[:3], "en")
            break
    tiers: list[Optional[str]] = []
    for line in lines:
        if line.startswith("*"):
            tiers.append(line)
        elif line.startswith("\t") and tiers and tiers[-1] is not None:
            tiers[-1] += " " + line.strip()
        elif line.startswith(("%", "@")):
            tiers.append(None)
    utts = []
    for tier in tiers:
        if not tier or ":" not in tier:
            continue
        head, body = tier.split(":", 1)
        marks = BULLET.findall(body)
        if not marks:
            continue
        start, end = int(marks[-1][0]) / 1000, int(marks[-1][1]) / 1000
        if end <= start:
            continue
        text, n_es, n_en, unint = clean_chat_line(body, default, conventions)
        utts.append(Utterance(head[1:].strip(), start, end, text, n_es, n_en, unint, body))
    utts.sort(key=lambda u: u.start)
    return utts, default


# --------------------------------------------------------------------------- clip building

def build_clips(utts: list[Utterance], rng: random.Random, max_sec: float = 28.0, min_sec: float = 1.0,
                max_gap: float = 1.5, max_crosstalk: float = 0.2) -> list[dict]:
    """Merge consecutive same-speaker utterances into clips of random length up to max_sec."""
    by_speaker: dict[str, list[Utterance]] = defaultdict(list)
    for u in utts:
        by_speaker[u.speaker].append(u)
    clips = []
    for speaker, own in by_speaker.items():
        others = [(u.start, u.end) for u in utts if u.speaker != speaker]
        i = 0
        while i < len(own):
            target = rng.uniform(min(6.0, max_sec), max_sec)
            group = [own[i]]
            j = i + 1
            while (j < len(own) and own[j].start - group[-1].end <= max_gap
                   and own[j].end - group[0].start <= target):
                group.append(own[j])
                j += 1
            i = j
            start, end = group[0].start, group[-1].end
            dur = end - start
            if dur < min_sec or dur > max_sec + 2 or any(u.unintelligible for u in group):
                continue
            text = " ".join(u.text for u in group if u.text).strip()
            if not text:
                continue
            overlap = sum(max(0.0, min(end, e) - max(start, s)) for s, e in others)
            if overlap / dur > max_crosstalk:
                continue
            n_es, n_en = sum(u.n_es for u in group), sum(u.n_en for u in group)
            clips.append({"speaker": speaker, "start": start, "end": end, "text": text,
                          "n_es": n_es, "n_en": n_en})
    return clips


def load_audio(path: Path) -> np.ndarray:
    try:
        from faster_whisper.audio import decode_audio

        return decode_audio(str(path), sampling_rate=SR)
    except ImportError:
        import librosa

        return librosa.load(str(path), sr=SR, mono=True)[0].astype(np.float32)


def process_conversation(job: tuple) -> list[dict]:
    cha, audio_path, out_dir, seed, max_sec, pad = job
    utts, default = parse_chat(Path(cha))
    rng = random.Random(f"{seed}-{Path(cha).stem}")
    clips = build_clips(utts, rng, max_sec=max_sec)
    if not clips:
        return []
    audio = load_audio(Path(audio_path))
    rows = []
    for k, c in enumerate(clips):
        lo = max(0, int((c["start"] - pad) * SR))
        hi = min(len(audio), int((c["end"] + pad) * SR))
        if hi - lo < SR * 0.5:
            continue
        name = f"{Path(cha).stem}_{k:04d}.flac"
        sf.write(Path(out_dir) / "clips" / name, audio[lo:hi], SR)
        lang = "es" if c["n_es"] > c["n_en"] else "en" if c["n_en"] > c["n_es"] else default
        rows.append({"audio": str(Path(out_dir) / "clips" / name), "text": c["text"], "language": lang,
                     "duration": round((hi - lo) / SR, 3), "source": "miami", "speaker": c["speaker"],
                     "conversation": Path(cha).stem, "n_es": c["n_es"], "n_en": c["n_en"]})
    return rows


# --------------------------------------------------------------------------- dev set

def dev_target(transcript: str) -> str:
    """Training target from a dev reference: drop [annotations] and (?) markers, keep the rest."""
    s = re.sub(r"\[[^\]]*\]", " ", str(transcript))
    s = re.sub(r"\(\?+\)", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def prepare_dev(dev_dir: Path, out_dir: Path, seed: int) -> pd.DataFrame:
    meta_path = next((p for p in (dev_dir / "metadata.tsv", dev_dir / "metadata.csv") if p.exists()), None)
    if meta_path is None:
        raise SystemExit(f"{dev_dir}: no metadata.tsv / metadata.csv with audio_filename,transcript")
    meta = pd.read_csv(meta_path, sep="\t" if meta_path.suffix == ".tsv" else ",", keep_default_na=False)
    rows = []
    for r in meta.itertuples(index=False):
        src = dev_dir / "clips" / r.audio_filename
        audio = load_audio(src)
        dst = out_dir / "clips" / f"dev_{Path(r.audio_filename).stem}.flac"
        sf.write(dst, audio, SR)
        rows.append({"audio": str(dst), "text": dev_target(r.transcript), "language": "",
                     "duration": round(len(audio) / SR, 3), "source": "dev",
                     "speaker": str(getattr(r, "speaker", "") or ""), "conversation": "dev",
                     "audio_filename": r.audio_filename, "original_audio": str(src),
                     "transcript": r.transcript})
    dev = pd.DataFrame(rows)
    # Split in half: by speaker when speakers are given, so halves share no voices.
    rng = random.Random(seed)
    if dev["speaker"].str.len().gt(0).all() and dev["speaker"].nunique() >= 4:
        speakers = sorted(dev["speaker"].unique())
        rng.shuffle(speakers)
        half, acc, target = set(), 0.0, dev["duration"].sum() / 2
        for s in speakers:
            if acc < target:
                half.add(s)
                acc += dev.loc[dev["speaker"] == s, "duration"].sum()
        dev["dev_split"] = np.where(dev["speaker"].isin(half), "a", "b")
    else:
        order = list(range(len(dev)))
        rng.shuffle(order)
        dev["dev_split"] = "b"
        dev.loc[order[: len(dev) // 2], "dev_split"] = "a"
    return dev


def language_of_text(text: str, table: dict[str, float]) -> str:
    """Majority language of a dev transcript, from P(Spanish) of words seen in Miami."""
    ps = [table[w] for w in re.findall(r"[^\W\d_]+(?:'[^\W\d_]+)?", text.lower()) if w in table]
    return "es" if ps and np.mean(ps) > 0.5 else "en"


# --------------------------------------------------------------------------- main

def audit(miami_text: pd.Series, dev_text: pd.Series) -> list[str]:
    """Per-1k-word frequencies of spelling variants in Miami targets vs dev references."""
    def freqs(texts):
        c = Counter(w for t in texts for w in re.findall(r"[^\W\d_]+'?[^\W\d_]*", str(t).lower()))
        return c, max(sum(c.values()), 1)
    (cm, nm), (cd, nd) = freqs(miami_text), freqs(dev_text)
    lines = []
    for a, b in AUDIT_PAIRS:
        lines.append(f"{a:>6}/{b:<8} Miami {1000 * cm[a] / nm:6.2f}/{1000 * cm[b] / nm:6.2f}   "
                     f"dev {1000 * cd[a] / nd:6.2f}/{1000 * cd[b] / nd:6.2f}  (per 1k words)")
    return lines


def main(argv=None) -> dict:
    ap = argparse.ArgumentParser(description=__doc__.split("\n\n")[0])
    ap.add_argument("--miami", required=True, help="dir with chat/*.cha and audios/*.mp3")
    ap.add_argument("--dev", required=True, help="dir with clips/ and metadata.tsv")
    ap.add_argument("--out", required=True)
    ap.add_argument("--holdout-frac", type=float, default=0.08, help="share of Miami hours held out by conversation")
    ap.add_argument("--max-sec", type=float, default=28.0)
    ap.add_argument("--pad", type=float, default=0.15, help="seconds of audio kept around each clip")
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--limit-conversations", type=int, help="only the first N conversations (quick tests)")
    args = ap.parse_args(argv)

    miami, dev_dir, out = Path(args.miami), Path(args.dev), Path(args.out)
    (out / "clips").mkdir(parents=True, exist_ok=True)
    chas = sorted((miami / "chat").glob("*.cha"))
    audio_for = {p.stem: p for p in (miami / "audios").glob("*") if p.suffix.lower() in (".mp3", ".wav", ".flac", ".ogg")}
    jobs = [(str(c), str(audio_for[c.stem]), str(out), args.seed, args.max_sec, args.pad)
            for c in chas if c.stem in audio_for]
    if args.limit_conversations:
        jobs = jobs[: args.limit_conversations]
    if not jobs:
        raise SystemExit(f"{miami}: found no chat/*.cha with a matching audios/<name>.mp3")
    print(f"Miami: {len(jobs)} conversations with audio ({len(chas)} transcripts)")
    rows = []
    with ProcessPoolExecutor(max_workers=max(1, args.workers)) as pool:
        for i, part in enumerate(pool.map(process_conversation, jobs)):
            rows.extend(part)
            if (i + 1) % 10 == 0 or i + 1 == len(jobs):
                print(f"  {i + 1}/{len(jobs)} conversations, {len(rows)} clips")
    miami_df = pd.DataFrame(rows)
    if miami_df.empty:
        raise SystemExit("no Miami clips survived filtering")

    # Conversation-level holdout (~holdout_frac of hours).
    rng = random.Random(args.seed)
    hours = miami_df.groupby("conversation")["duration"].sum()
    convs = sorted(hours.index)
    rng.shuffle(convs)
    held, acc = set(), 0.0
    for c in convs:
        if acc >= args.holdout_frac * hours.sum() or len(held) >= len(convs) - 1:
            break
        held.add(c)
        acc += hours[c]
    holdout = miami_df[miami_df["conversation"].isin(held)]
    train = miami_df[~miami_df["conversation"].isin(held)]

    dev = prepare_dev(dev_dir, out, args.seed)
    word_lang = Counter()
    for r in train.itertuples():
        for w in re.findall(r"[^\W\d_]+(?:'[^\W\d_]+)?", r.text.lower()):
            word_lang[(w, r.language)] += 1
    table = {}
    for (w, lang), n in word_lang.items():
        es, en = word_lang[(w, "es")], word_lang[(w, "en")]
        if es + en >= 3:
            table[w] = es / (es + en)
    dev["language"] = [language_of_text(t, table) for t in dev["text"]]

    cols = ["audio", "text", "language", "duration", "source", "speaker", "conversation"]
    train[cols].to_csv(out / "miami_train.csv", index=False)
    holdout[cols].to_csv(out / "miami_holdout.csv", index=False)
    dev_cols = cols + ["audio_filename", "original_audio", "transcript", "dev_split"]
    dev[dev_cols].to_csv(out / "dev_all.csv", index=False)
    for part in ("a", "b"):
        dev[dev["dev_split"] == part][dev_cols].to_csv(out / f"dev_{part}.csv", index=False)
    dev[["audio_filename", "transcript"]].to_csv(out / "gold_dev_all.csv", index=False)
    dev[dev["dev_split"] == "b"][["audio_filename", "transcript"]].to_csv(out / "gold_dev_b.csv", index=False)

    stats = {
        "miami_train_hours": round(train["duration"].sum() / 3600, 2),
        "miami_train_clips": int(len(train)),
        "miami_holdout_hours": round(holdout["duration"].sum() / 3600, 2),
        "miami_holdout_clips": int(len(holdout)),
        "miami_language_mix": {k: int(v) for k, v in train["language"].value_counts().items()},
        "dev_clips": int(len(dev)),
        "dev_minutes": round(dev["duration"].sum() / 60, 1),
        "dev_split_clips": {k: int(v) for k, v in dev["dev_split"].value_counts().items()},
        "dev_language_mix": {k: int(v) for k, v in dev["language"].value_counts().items()},
        "convention_audit": audit(train["text"], dev["transcript"]),
    }
    (out / "stats.json").write_text(json.dumps(stats, indent=2, ensure_ascii=False))
    print(json.dumps({k: v for k, v in stats.items() if k != "convention_audit"}, indent=2))
    print("Spelling audit (check the conventions before training):")
    for line in stats["convention_audit"]:
        print("  " + line)
    return stats


if __name__ == "__main__":
    main()


In [ ]:
%%writefile train_whisper.py
#!/usr/bin/env python3
"""LoRA fine-tuning of Whisper on clips made by prepare_data.py.

Runs on one GPU, on several with torchrun, or on CPU for smoke tests. Precision follows the GPU:
bf16 on Ampere or newer, fp16 on older cards such as Kaggle's T4 and P100. Every ``--eval-steps``
it decodes a fixed sample of the eval set and scores it with the official normaliser, keeps the
checkpoint with the lowest WER, and writes it to ``OUTPUT/best_adapter``.

    torchrun --standalone --nproc_per_node=2 train_whisper.py \\
        --train data/miami_train.csv,data/dev_a.csv --eval data/miami_holdout.csv \\
        --scorer lost-in-transcription-runtime/score.py --output /tmp/runs/lv3

Defaults follow what worked for another team on this data (LoRA rank 64 on the upper half of the
encoder plus the decoder, lr 1e-4, effective batch 48); they saw gains flatten after 100-200 steps.
"""

from __future__ import annotations

import argparse
import importlib.util
import io
import json
import os
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import torch
from peft import LoraConfig, get_peft_model
from rapidfuzz.distance import Levenshtein
from transformers import (Seq2SeqTrainer, Seq2SeqTrainingArguments, TrainerCallback,
                          WhisperForConditionalGeneration, WhisperProcessor)

SR = 16000
MAX_SAMPLES = 30 * SR
LORA_LEAVES = {"q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"}


# --------------------------------------------------------------------------- data

def codec_roundtrip(audio: np.ndarray, rng: random.Random) -> np.ndarray:
    """Encode to Opus at a low bitrate and back, like a phone voice note."""
    import av

    buf = io.BytesIO()
    out = av.open(buf, "w", format="ogg")
    stream = out.add_stream("libopus", rate=SR, options={"compression_level": "3", "application": "voip"})
    stream.layout = "mono"
    stream.bit_rate = rng.choice([12000, 16000, 24000, 32000])
    pcm = (np.clip(audio, -1, 1) * 32767).astype(np.int16)
    for i in range(0, len(pcm), 320):  # 20 ms frames
        chunk = pcm[i:i + 320]
        if len(chunk) < 320:
            chunk = np.pad(chunk, (0, 320 - len(chunk)))
        frame = av.AudioFrame.from_ndarray(chunk[None, :], format="s16", layout="mono")
        frame.sample_rate, frame.pts = SR, i
        for packet in stream.encode(frame):
            out.mux(packet)
    for packet in stream.encode(None):
        out.mux(packet)
    out.close()
    buf.seek(0)
    decoded = []
    with av.open(buf) as inp:
        resampler = av.AudioResampler(format="s16", layout="mono", rate=SR)
        for frame in inp.decode(audio=0):
            decoded.extend(f.to_ndarray().reshape(-1) for f in resampler.resample(frame))
        decoded.extend(f.to_ndarray().reshape(-1) for f in resampler.resample(None))
    y = np.concatenate(decoded).astype(np.float32) / 32768 if decoded else np.zeros_like(audio)
    return y[: len(audio)] if len(y) >= len(audio) else np.pad(y, (0, len(audio) - len(y)))


class ClipDataset(torch.utils.data.Dataset):
    """Rows of audio,text,language[,source] -> Whisper log-mel features and label ids."""

    def __init__(self, df: pd.DataFrame, processor: WhisperProcessor, augment_prob: float = 0.0):
        self.rows = df.to_dict("records")
        self.fe = processor.feature_extractor
        self.tok = processor.tokenizer
        self.augment_prob = augment_prob

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        audio, sr = sf.read(r["audio"], dtype="float32", always_2d=False)
        if audio.ndim > 1:
            audio = audio.mean(axis=1)
        if sr != SR:
            import librosa

            audio = librosa.resample(audio, orig_sr=sr, target_sr=SR)
        audio = audio[:MAX_SAMPLES]
        if self.augment_prob and r.get("source") == "miami" and random.random() < self.augment_prob:
            rng = random.Random(random.random())
            audio = codec_roundtrip(audio, rng) * (10 ** (rng.uniform(-6, 6) / 20))
        feats = self.fe(audio, sampling_rate=SR).input_features[0]
        lang = r.get("language") if isinstance(r.get("language"), str) and r.get("language") else "en"
        self.tok.set_prefix_tokens(language=lang, task="transcribe", predict_timestamps=False)
        labels = self.tok(str(r["text"])).input_ids[:440]
        return {"input_features": feats, "labels": labels}


class Collator:
    def __init__(self, decoder_start_token_id: int, dtype: torch.dtype):
        self.start = decoder_start_token_id
        self.dtype = dtype

    def __call__(self, features):
        feats = torch.tensor(np.stack([f["input_features"] for f in features]), dtype=self.dtype)
        width = max(len(f["labels"]) for f in features)
        labels = torch.full((len(features), width), -100, dtype=torch.long)
        for i, f in enumerate(features):
            labels[i, : len(f["labels"])] = torch.tensor(f["labels"])
        if (labels[:, 0] == self.start).all():  # the model prepends <|startoftranscript|> itself
            labels = labels[:, 1:]
        return {"input_features": feats, "labels": labels}


def read_rows(spec: str) -> pd.DataFrame:
    frames = [pd.read_csv(p, keep_default_na=False) for p in spec.split(",") if p]
    df = pd.concat(frames, ignore_index=True)
    return df[df["text"].astype(str).str.strip() != ""].reset_index(drop=True)


# --------------------------------------------------------------------------- model

def lora_targets(model, encoder_from_layer: int, decoder: bool) -> list[str]:
    """Attention and MLP projections of encoder layers >= encoder_from_layer (and the decoder)."""
    names = []
    for name, module in model.named_modules():
        parts = name.split(".")
        if not isinstance(module, torch.nn.Linear) or "layers" not in parts or parts[-1] not in LORA_LEAVES:
            continue
        side, idx = parts[parts.index("layers") - 1], int(parts[parts.index("layers") + 1])
        if (side == "encoder" and idx >= encoder_from_layer) or (side == "decoder" and decoder):
            names.append(name)
    return names


def load_normalizer(path: str):
    spec = importlib.util.spec_from_file_location("lit_official_score", path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module.normalize_text


def make_metrics(tokenizer, normalize):
    def words(text):
        return [w for w in re.sub(r"\s\s+", " ", normalize(text)).strip().split(" ") if w]

    def compute(pred):
        pred_ids = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
        pred_ids = np.where(pred_ids < 0, tokenizer.pad_token_id, pred_ids)
        label_ids = np.where(pred.label_ids < 0, tokenizer.pad_token_id, pred.label_ids)
        hyps = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
        refs = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
        errors = total = 0
        for h, r in zip(hyps, refs):
            rw = words(r)
            errors += Levenshtein.distance(rw, words(h))
            total += len(rw)
        return {"wer": errors / max(total, 1)}

    return compute


class PrintEval(TrainerCallback):
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if state.is_world_process_zero and metrics:
            print(f"[eval] step {state.global_step}: WER {metrics.get('eval_wer', float('nan')):.4f}", flush=True)


# --------------------------------------------------------------------------- main

def main(argv=None) -> None:
    ap = argparse.ArgumentParser(description=__doc__.split("\n\n")[0])
    ap.add_argument("--train", required=True, help="comma-separated CSVs from prepare_data.py")
    ap.add_argument("--eval", required=True, help="CSV used to pick the best checkpoint")
    ap.add_argument("--scorer", required=True, help="official score.py (for its normaliser)")
    ap.add_argument("--output", required=True)
    ap.add_argument("--model", default="openai/whisper-large-v3")
    ap.add_argument("--max-steps", type=int, default=300)
    ap.add_argument("--eval-steps", type=int, default=50)
    ap.add_argument("--eval-samples", type=int, default=160, help="eval clips decoded at each evaluation")
    ap.add_argument("--lr", type=float, default=1e-4)
    ap.add_argument("--warmup-steps", type=int, default=30)
    ap.add_argument("--batch-size", type=int, default=8, help="per-device micro-batch")
    ap.add_argument("--effective-batch", type=int, default=48)
    ap.add_argument("--lora-r", type=int, default=64)
    ap.add_argument("--lora-alpha", type=int, default=64)
    ap.add_argument("--lora-dropout", type=float, default=0.0)
    ap.add_argument("--encoder-from-layer", type=int, default=16)
    ap.add_argument("--no-decoder-lora", action="store_true")
    ap.add_argument("--augment-prob", type=float, default=0.5, help="Opus round trip on Miami clips")
    ap.add_argument("--workers", type=int, help="dataloader workers per process (default: CPUs / processes)")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--no-eval-on-start", action="store_true", help="skip the step-0 (zero-shot) evaluation")
    ap.add_argument("--resume", action="store_true", help="continue from the last checkpoint in --output")
    ap.add_argument("--cpu", action="store_true", help="smoke tests without a GPU")
    args = ap.parse_args(argv)

    world = int(os.environ.get("WORLD_SIZE", "1"))
    gpu = torch.cuda.is_available() and not args.cpu
    # is_bf16_supported() also counts emulation (True on a T4); real bf16 needs Ampere or newer
    bf16 = gpu and torch.cuda.get_device_capability(0)[0] >= 8
    fp16 = gpu and not bf16
    dtype = torch.bfloat16 if bf16 else torch.float16 if fp16 else torch.float32
    accum = max(1, round(args.effective_batch / (args.batch_size * world)))
    workers = args.workers if args.workers is not None else max(1, (os.cpu_count() or 2) // world)
    random.seed(args.seed)

    processor = WhisperProcessor.from_pretrained(args.model)
    model = WhisperForConditionalGeneration.from_pretrained(args.model, dtype=dtype)
    model.config.use_cache = False
    model.generation_config.forced_decoder_ids = None
    model.generation_config.language = None  # detect the language, as at inference
    model.generation_config.task = "transcribe"
    targets = lora_targets(model, args.encoder_from_layer, not args.no_decoder_lora)
    model = get_peft_model(model, LoraConfig(r=args.lora_r, lora_alpha=args.lora_alpha,
                                             lora_dropout=args.lora_dropout, target_modules=targets, bias="none"))

    train_df = read_rows(args.train)
    eval_df = read_rows(args.eval).sample(frac=1.0, random_state=args.seed).head(args.eval_samples)
    rank0 = int(os.environ.get("RANK", "0")) == 0
    if rank0:
        model.print_trainable_parameters()
        print(f"train clips {len(train_df)} ({train_df['duration'].sum() / 3600:.1f} h) | eval clips {len(eval_df)} | "
              f"{'bf16' if bf16 else 'fp16' if fp16 else 'fp32'} | {world} process(es) x batch {args.batch_size} "
              f"x accum {accum} = {args.batch_size * world * accum}", flush=True)

    training_args = Seq2SeqTrainingArguments(
        output_dir=args.output,
        per_device_train_batch_size=args.batch_size,
        per_device_eval_batch_size=max(args.batch_size, 8),
        gradient_accumulation_steps=accum,
        learning_rate=args.lr,
        warmup_steps=args.warmup_steps,
        max_steps=args.max_steps,
        lr_scheduler_type="linear",
        bf16=bf16,
        fp16=fp16,
        use_cpu=not gpu,
        gradient_checkpointing=gpu,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        eval_strategy="steps",
        eval_steps=args.eval_steps,
        eval_on_start=not args.no_eval_on_start,
        save_strategy="steps",
        save_steps=args.eval_steps,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="wer",
        greater_is_better=False,
        predict_with_generate=True,
        generation_max_length=225,
        logging_steps=10,
        report_to="none",
        dataloader_num_workers=workers if gpu else 0,
        remove_unused_columns=False,
        label_names=["labels"],
        ddp_find_unused_parameters=False,
        seed=args.seed,
    )
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=ClipDataset(train_df, processor, args.augment_prob),
        eval_dataset=ClipDataset(eval_df, processor),
        data_collator=Collator(model.config.decoder_start_token_id, dtype),
        compute_metrics=make_metrics(processor.tokenizer, load_normalizer(args.scorer)),
        callbacks=[PrintEval()],
    )
    resume = args.resume and any(Path(args.output).glob("checkpoint-*"))
    trainer.train(resume_from_checkpoint=True if resume else None)

    if trainer.is_world_process_zero():
        best_dir = Path(args.output) / "best_adapter"
        trainer.model.save_pretrained(best_dir)
        processor.save_pretrained(best_dir)
        history = [h for h in trainer.state.log_history if "eval_wer" in h]
        summary = {
            "base_model": args.model,
            "best_checkpoint": trainer.state.best_model_checkpoint,
            "best_eval_wer": trainer.state.best_metric,
            "eval_history": [{"step": h["step"], "wer": h["eval_wer"]} for h in history],
            "train_loss": [{"step": h["step"], "loss": h["loss"]} for h in trainer.state.log_history if "loss" in h],
            "args": vars(args),
        }
        (Path(args.output) / "summary.json").write_text(json.dumps(summary, indent=2))
        print(f"best eval WER {summary['best_eval_wer']} at {summary['best_checkpoint']}; adapter -> {best_dir}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile export_whisper.py
#!/usr/bin/env python3
"""Merge a Whisper LoRA adapter and convert the result to CTranslate2 for faster-whisper.

The output directory is what the competition runtime loads (faster-whisper 1.2.1 on
ctranslate2 4.8.2), and it includes tokenizer.json and preprocessor_config.json so nothing is
downloaded at inference time.

    python export_whisper.py --adapter /tmp/runs/lv3/best_adapter --out-ct2 /kaggle/working/model
"""

from __future__ import annotations

import argparse
import json
import shutil
import tempfile
from pathlib import Path

import torch
from peft import PeftConfig, PeftModel
from transformers import WhisperForConditionalGeneration, WhisperProcessor, WhisperTokenizerFast


def main(argv=None) -> None:
    ap = argparse.ArgumentParser(description=__doc__.split("\n\n")[0])
    ap.add_argument("--adapter", required=True, help="LoRA adapter directory (train_whisper.py best_adapter)")
    ap.add_argument("--base", help="base model (default: the one recorded in the adapter)")
    ap.add_argument("--out-ct2", required=True)
    ap.add_argument("--out-hf", help="also keep the merged Hugging Face model here")
    ap.add_argument("--quantization", default="float16", help="float16 | int8_float16 | int8 | float32")
    args = ap.parse_args(argv)

    base = args.base or PeftConfig.from_pretrained(args.adapter).base_model_name_or_path
    hf_dir = Path(args.out_hf) if args.out_hf else Path(tempfile.mkdtemp(prefix="merged_"))
    model = WhisperForConditionalGeneration.from_pretrained(base, dtype=torch.float32)
    model = PeftModel.from_pretrained(model, args.adapter).merge_and_unload()
    model.generation_config.forced_decoder_ids = None
    model.save_pretrained(hf_dir, safe_serialization=True)
    WhisperProcessor.from_pretrained(base).save_pretrained(hf_dir)
    if not (hf_dir / "tokenizer.json").exists():  # faster-whisper needs the fast tokenizer file offline
        WhisperTokenizerFast.from_pretrained(base).save_pretrained(hf_dir)

    from ctranslate2.converters import TransformersConverter

    out = Path(args.out_ct2)
    TransformersConverter(str(hf_dir), copy_files=["tokenizer.json", "preprocessor_config.json"]).convert(
        str(out), quantization=args.quantization, force=True)
    (out / "export_info.json").write_text(json.dumps({"base": base, "adapter": str(args.adapter),
                                                     "quantization": args.quantization}, indent=2))
    if not args.out_hf:
        shutil.rmtree(hf_dir, ignore_errors=True)

    from faster_whisper import WhisperModel  # fail here, not on the platform, if the export is broken

    WhisperModel(str(out), device="cpu", compute_type="int8")
    size = sum(f.stat().st_size for f in out.rglob("*") if f.is_file()) / 1e9
    print(f"CTranslate2 model ({args.quantization}, {size:.2f} GB) -> {out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile run_systems.py
#!/usr/bin/env python3
"""Transcribe a labelled set with several ASR approaches, one prediction CSV per approach.

Each ``--system`` is ``NAME=FAMILY:MODEL[?key=value&key=value...]``:

  whisper    faster-whisper / CTranslate2 (hub size like large-v3, a CT2 repo id, or a local
             fine-tuned export). Options: lang=auto|es|en (default auto), beam=5, vad=0|1,
             prev=1|0 (condition on previous text), multi=0|1 (re-detect language per 30 s
             window), batch=N (batched VAD pipeline), prompt_file=PATH (initial prompt).
  qwen3-asr  Qwen3-ASR through the qwen-asr package. Options: lang=auto|es|en, backend=vllm|
             transformers (default: vllm when a GPU and vLLM are present), batch=N,
             max_new_tokens=2048, gpu_mem=0.85, context=A.csv+B.csv (per-clip context built from
             other systems' predictions, e.g. the Spanish- and English-forced Whisper outputs),
             context_file=PATH (one context text for every clip, e.g. hotwords).

Every system runs in its own subprocess so GPU memory is released between systems, and a
system whose CSV already exists is skipped unless --overwrite. Output per system:
``OUT/NAME.csv`` (submission format, so ``compare.py OUT/*.csv`` scores every system), plus
``OUT/info/NAME.timing.json`` (load time, decode time, real-time factor),
``OUT/info/NAME.details.csv`` (Qwen's detected language) and ``OUT/info/meta.csv`` (clip
durations for ``compare.py --meta OUT/info/meta.csv --by duration_s:10,30,60``).

Example (Colab / GPU)::

    python run_systems.py --manifest data/submission_format.csv --clips data/clips --out-dir preds \\
        --system "wh_auto=whisper:large-v3" --system "wh_es=whisper:large-v3?lang=es" \\
        --system "wh_en=whisper:large-v3?lang=en" \\
        --system "qwen_auto=qwen3-asr:Qwen/Qwen3-ASR-1.7B" \\
        --system "qwen_merge=qwen3-asr:Qwen/Qwen3-ASR-1.7B?context=preds/wh_es.csv+preds/wh_en.csv"
"""

from __future__ import annotations

import argparse
import importlib.util
import json
import re
import subprocess
import sys
import time
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd

SAMPLE_RATE = 16000
QWEN_LANGS = {"es": "Spanish", "en": "English"}


# --------------------------------------------------------------------------- helpers

def parse_system(spec: str) -> dict:
    """``NAME=FAMILY:MODEL?k=v&k=v`` -> {"name", "family", "model", "opts"}."""
    if "=" not in spec or ":" not in spec.split("=", 1)[1]:
        raise SystemExit(f"--system {spec!r}: expected NAME=FAMILY:MODEL[?key=value&...]")
    name, rest = spec.split("=", 1)
    family, rest = rest.split(":", 1)
    model, _, query = rest.partition("?")
    family = family.strip().lower()
    if family not in ("whisper", "qwen3-asr"):
        raise SystemExit(f"--system {spec!r}: unknown family {family!r} (whisper, qwen3-asr)")
    opts = {}
    for item in filter(None, query.split("&")):  # no URL decoding: '+' separates context CSVs
        key, _, value = item.partition("=")
        opts[key.strip()] = value.strip()
    return {"name": name.strip(), "family": family, "model": model.strip(), "opts": opts}


def load_audio(path: Path) -> np.ndarray:
    """Mono 16 kHz float32, decoded with PyAV (mp3/ogg/opus/m4a/wav) or librosa as a fallback."""
    try:
        from faster_whisper.audio import decode_audio

        return decode_audio(str(path), sampling_rate=SAMPLE_RATE)
    except ImportError:
        import librosa

        audio, _ = librosa.load(str(path), sr=SAMPLE_RATE, mono=True)
        return audio.astype(np.float32)


def cuda_available() -> bool:
    try:
        import torch

        return torch.cuda.is_available()
    except ImportError:
        try:
            import ctranslate2

            return ctranslate2.get_cuda_device_count() > 0
        except ImportError:
            return False


def flag(opts: dict, key: str, default: bool) -> bool:
    return str(opts.get(key, int(default))).lower() in ("1", "true", "yes")


def read_manifest(path: Path, limit: Optional[int]) -> list[str]:
    names = pd.read_csv(path)["audio_filename"].astype(str).tolist()
    return names[:limit] if limit else names


def build_context(opts: dict, names: list[str]) -> list[str]:
    """Per-clip context: the given prediction CSVs' transcripts (one per line) and/or a fixed text."""
    fixed = Path(opts["context_file"]).read_text().strip() if "context_file" in opts else ""
    per_clip = [[] for _ in names]
    for csv in filter(None, opts.get("context", "").split("+")):
        pred = pd.read_csv(csv).set_index("audio_filename")["transcript"].fillna("").astype(str)
        for i, n in enumerate(names):
            per_clip[i].append(pred.get(n, ""))
    return ["\n".join(filter(None, [fixed, *parts])) for parts in per_clip]


# --------------------------------------------------------------------------- families

def run_whisper(system: dict, audios: list[np.ndarray], log) -> list[str]:
    from faster_whisper import BatchedInferencePipeline, WhisperModel

    opts = system["opts"]
    device = "cuda" if cuda_available() else "cpu"

    def load(dev):
        return WhisperModel(system["model"], device=dev,
                            compute_type=opts.get("compute_type", "float16" if dev == "cuda" else "int8"))

    t0 = time.time()
    model = load(device)
    log(f"model loaded in {time.time() - t0:.0f}s", load_seconds=time.time() - t0)
    lang = opts.get("lang", "auto")
    kwargs = dict(
        language=None if lang == "auto" else lang,
        beam_size=int(opts.get("beam", 5)),
        condition_on_previous_text=flag(opts, "prev", True),
        multilingual=flag(opts, "multi", False),
        initial_prompt=Path(opts["prompt_file"]).read_text().strip() if "prompt_file" in opts else None,
    )
    batch = int(opts.get("batch", 0))
    pipe = BatchedInferencePipeline(model) if batch > 0 else None
    def decode(audio):
        if pipe:  # the batched pipeline always segments with VAD
            segments, _ = pipe.transcribe(audio, batch_size=batch, vad_filter=True, **kwargs)
        else:
            segments, _ = model.transcribe(audio, vad_filter=flag(opts, "vad", False), **kwargs)
        return " ".join(s.text.strip() for s in segments).strip()  # errors surface while iterating

    texts = []
    for i, audio in enumerate(audios):
        try:
            texts.append(decode(audio))
        except RuntimeError as exc:
            # CTranslate2 opens cuBLAS/cuDNN at the first GPU computation, so a missing CUDA 12
            # library only shows up here. Fall back to CPU instead of losing the run.
            if device != "cuda" or i > 0 or not re.search(r"cublas|cudnn|cuda", str(exc), re.I):
                raise
            log(f"GPU decoding failed ({exc}); falling back to CPU int8")
            device = "cpu"
            model = load(device)
            pipe = BatchedInferencePipeline(model) if batch > 0 else None
            texts.append(decode(audio))
        if (i + 1) % 25 == 0 or i + 1 == len(audios):
            log(f"{i + 1}/{len(audios)} clips")
    return texts


def run_qwen(system: dict, audios: list[np.ndarray], names: list[str], log) -> tuple[list[str], list[str]]:
    from qwen_asr import Qwen3ASRModel

    opts = system["opts"]
    gpu = cuda_available()
    backend = opts.get("backend")
    if backend is None:  # find_spec checks for vLLM without paying for its import
        backend = "vllm" if gpu and importlib.util.find_spec("vllm") else "transformers"
    max_new = int(opts.get("max_new_tokens", 2048))  # qwen-asr's transformers default (512) truncates long clips
    t0 = time.time()
    if backend == "vllm":
        model = Qwen3ASRModel.LLM(model=system["model"], gpu_memory_utilization=float(opts.get("gpu_mem", 0.85)),
                                  max_model_len=int(opts.get("max_model_len", 8192)),
                                  max_inference_batch_size=int(opts.get("batch", -1)), max_new_tokens=max_new)
    else:
        import torch

        model = Qwen3ASRModel.from_pretrained(
            system["model"], dtype=torch.bfloat16 if gpu else torch.float32,
            device_map="cuda:0" if gpu else "cpu",
            max_inference_batch_size=int(opts.get("batch", 16)), max_new_tokens=max_new)
    log(f"{backend} model loaded in {time.time() - t0:.0f}s", load_seconds=time.time() - t0)
    lang = opts.get("lang", "auto")
    language = None if lang == "auto" else QWEN_LANGS.get(lang, lang)
    contexts = build_context(opts, names)
    texts, langs = [], []
    step = 64
    for start in range(0, len(audios), step):
        chunk = slice(start, start + step)
        results = model.transcribe(audio=[(a, SAMPLE_RATE) for a in audios[chunk]],
                                   context=contexts[chunk], language=language)
        texts.extend(r.text.strip() for r in results)
        langs.extend(r.language for r in results)
        log(f"{min(start + step, len(audios))}/{len(audios)} clips")
    return texts, langs


# --------------------------------------------------------------------------- driver

def worker(system: dict, args) -> None:
    """Transcribe every clip with one system and write NAME.csv + NAME.timing.json."""
    out_dir = Path(args.out_dir)
    info_dir = out_dir / "info"
    names = read_manifest(Path(args.manifest), args.limit)
    load = {"seconds": 0.0}

    def log(msg, load_seconds=None):
        if load_seconds is not None:
            load["seconds"] = load_seconds
        print(f"[{system['name']}] {msg}", flush=True)

    t0 = time.time()
    audios = [load_audio(Path(args.clips) / n) for n in names]
    audio_seconds = float(sum(len(a) for a in audios) / SAMPLE_RATE)
    log(f"decoded {len(audios)} clips ({audio_seconds / 60:.1f} min) in {time.time() - t0:.0f}s")

    t1 = time.time()
    details = None
    if system["family"] == "whisper":
        texts = run_whisper(system, audios, log)
    else:
        texts, langs = run_qwen(system, audios, names, log)
        details = pd.DataFrame({"audio_filename": names, "language": langs})
    decode_seconds = time.time() - t1 - load["seconds"]

    pd.DataFrame({"audio_filename": names, "transcript": texts}).to_csv(out_dir / f"{system['name']}.csv", index=False)
    if details is not None:
        details.to_csv(info_dir / f"{system['name']}.details.csv", index=False)
    timing = {
        "system": system, "clips": len(names), "audio_seconds": audio_seconds,
        "load_seconds": load["seconds"], "decode_seconds": decode_seconds,
        "rtfx": audio_seconds / max(decode_seconds, 1e-9), "gpu": cuda_available(),
    }
    (info_dir / f"{system['name']}.timing.json").write_text(json.dumps(timing, indent=2))
    log(f"decoded {audio_seconds / 60:.1f} min of audio in {decode_seconds:.0f}s "
        f"({timing['rtfx']:.1f}x real time; model load {load['seconds']:.0f}s)")


def write_meta(args) -> None:
    meta_path = Path(args.out_dir) / "info" / "meta.csv"
    names = read_manifest(Path(args.manifest), args.limit)
    if meta_path.exists() and not args.overwrite and set(names) <= set(pd.read_csv(meta_path)["audio_filename"]):
        return
    durations = [len(load_audio(Path(args.clips) / n)) / SAMPLE_RATE for n in names]
    pd.DataFrame({"audio_filename": names, "duration_s": durations}).to_csv(meta_path, index=False)


def main(argv=None) -> None:
    ap = argparse.ArgumentParser(description=__doc__.split("\n\n")[0])
    ap.add_argument("--manifest", required=True, help="CSV with an audio_filename column (submission_format.csv or gold)")
    ap.add_argument("--clips", required=True, help="directory holding the audio files")
    ap.add_argument("--out-dir", required=True)
    ap.add_argument("--system", action="append", default=[], help="NAME=FAMILY:MODEL[?key=value&...]; repeatable")
    ap.add_argument("--limit", type=int, help="only the first N clips (quick checks)")
    ap.add_argument("--overwrite", action="store_true", help="re-run systems whose CSV exists")
    ap.add_argument("--in-process", action="store_true", help="run systems in this process (no GPU isolation)")
    ap.add_argument("--worker", help=argparse.SUPPRESS)
    args = ap.parse_args(argv)

    (Path(args.out_dir) / "info").mkdir(parents=True, exist_ok=True)
    if args.worker:
        worker(json.loads(args.worker), args)
        return
    systems = [parse_system(s) for s in args.system]
    if not systems:
        ap.error("give at least one --system")
    if len({s["name"] for s in systems}) != len(systems):
        ap.error("system names must be unique")
    write_meta(args)
    failed = []
    for system in systems:
        if (Path(args.out_dir) / f"{system['name']}.csv").exists() and not args.overwrite:
            print(f"[{system['name']}] exists, skipping (use --overwrite)")
            continue
        if args.in_process:
            worker(system, args)
            continue
        cmd = [sys.executable, __file__, "--manifest", args.manifest, "--clips", args.clips,
               "--out-dir", args.out_dir, "--worker", json.dumps(system)]
        if args.limit:
            cmd += ["--limit", str(args.limit)]
        if subprocess.run(cmd).returncode != 0:
            failed.append(system["name"])
    if failed:
        sys.exit(f"failed systems: {', '.join(failed)}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile compare.py
#!/usr/bin/env python3
"""Compare ASR systems on a labelled set with the official Lost in Transcription metric.

Every prediction CSV (columns ``audio_filename,transcript``, i.e. submission format) is scored
against a reference CSV with the organisers' own normaliser, imported from ``score.py`` in
https://github.com/drivendataorg/lost-in-transcription-runtime, and the same word alignment
jiwer uses, so the WER printed here is the WER the platform would report on that set.

On top of the raw WER it reports what you need to choose between approaches on a small set:

* a bootstrap 95% confidence interval for each system,
* the *paired* difference to the best system (or ``--baseline``), with its own interval and the
  share of bootstrap resamples in which the system beats the baseline,
* substitution / deletion / insertion rates and how much WER is caused by casing alone,
* error rates on reference fillers, Spanish words, English words, and the first word after a
  language switch (a PIER-style code-switching diagnostic),
* the oracle WER (best system picked per clip) and word-level ROVER / MBR combinations,
* WER broken down by any metadata column (speaker, duration bucket, ...).

Example::

    python compare.py --ref dev_gold.csv --scorer ../lost-in-transcription-runtime/score.py \\
        preds/*.csv --combine all --out-dir results/dev
"""

from __future__ import annotations

import argparse
import importlib.util
import json
import os
import re
import sys
from collections import defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from typing import Callable, Optional, Sequence

import numpy as np
import pandas as pd
from rapidfuzz.distance import Levenshtein

FILLERS = {
    "uh", "um", "uhm", "umm", "er", "erm", "eh", "ah", "ahh", "oh", "mm", "mmm", "hm", "hmm",
    "mhm", "uh-huh", "huh", "em", "ehm",
}
SCORER_CANDIDATES = (  # a runtime checkout here, or next to this repo
    "lost-in-transcription-runtime/score.py",
    "../lost-in-transcription-runtime/score.py",
    "../../lost-in-transcription-runtime/score.py",
    "../../../lost-in-transcription-runtime/score.py",
)
_MULTISPACE = re.compile(r"\s\s+")
_SPANISH_CHARS = set("áéíóúüñ")


# --------------------------------------------------------------------------- scoring primitives

def find_scorer(explicit: Optional[str]) -> Path:
    """Locate the official score.py (argument, $LIT_SCORE_PY, or a sibling runtime checkout)."""
    here = Path(__file__).resolve().parent
    candidates = [explicit, os.environ.get("LIT_SCORE_PY")]
    candidates += [c for c in SCORER_CANDIDATES] + [str(here / c) for c in SCORER_CANDIDATES]
    for c in candidates:
        if c and Path(c).is_file():
            return Path(c)
    sys.exit(
        "Cannot find the official score.py. Clone "
        "https://github.com/drivendataorg/lost-in-transcription-runtime and pass "
        "--scorer path/to/score.py (or set LIT_SCORE_PY)."
    )


def load_normalizer(path: Path) -> Callable[[str], str]:
    """Import ``normalize_text`` from the official scorer so scoring can never drift from it."""
    spec = importlib.util.spec_from_file_location("lit_official_score", path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module.normalize_text


def tokenize(text: str) -> list[str]:
    """Split a normalised transcript into words exactly like jiwer's default WER transform."""
    return [w for w in _MULTISPACE.sub(" ", text).strip().split(" ") if w]


def align(ref: Sequence[str], hyp: Sequence[str]) -> tuple[int, int, int, list[tuple[Optional[int], Optional[int]]]]:
    """Levenshtein alignment as computed by jiwer (rapidfuzz opcodes over word ids).

    Returns (substitutions, deletions, insertions, pairs) where pairs lists
    (ref_index or None, hyp_index or None) in order.
    """
    vocab: dict[str, int] = {}
    r = [vocab.setdefault(w, len(vocab)) for w in ref]
    h = [vocab.setdefault(w, len(vocab)) for w in hyp]
    subs = dels = ins = 0
    pairs: list[tuple[Optional[int], Optional[int]]] = []
    for tag, i1, i2, j1, j2 in Levenshtein.opcodes(r, h):
        if tag == "equal":
            pairs.extend((i1 + k, j1 + k) for k in range(i2 - i1))
        elif tag == "replace":
            subs += i2 - i1  # jiwer's convention; rapidfuzz replace blocks are 1:1
            pairs.extend((i1 + k, j1 + k) for k in range(i2 - i1))
        elif tag == "delete":
            dels += i2 - i1
            pairs.extend((i1 + k, None) for k in range(i2 - i1))
        elif tag == "insert":
            ins += j2 - j1
            pairs.extend((None, j1 + k) for k in range(j2 - j1))
    return subs, dels, ins, pairs


def read_transcripts(path: Path, allow_missing_col: bool = False) -> pd.Series:
    """Read a submission-format CSV the way score.py does (default NA parsing, then fillna(""))."""
    df = pd.read_csv(path)
    if "audio_filename" not in df.columns or ("transcript" not in df.columns and not allow_missing_col):
        sys.exit(f"{path}: expected columns audio_filename,transcript; got {list(df.columns)}")
    dupes = df["audio_filename"][df["audio_filename"].duplicated()]
    if len(dupes):
        sys.exit(f"{path}: duplicate audio_filename rows, e.g. {dupes.iloc[0]}")
    return df.set_index("audio_filename")["transcript"].fillna("").astype(str)


# --------------------------------------------------------------------------- language tagging

def make_lang_tagger(lexicon_path: Optional[str], margin: float = 1.0) -> Optional[Callable[[str], str]]:
    """Return token -> 'es' | 'en' | 'amb' | 'filler' | 'num' | 'unk', or None if no resource.

    Uses an optional lexicon CSV (columns word,lang with lang in {es,en,amb}) first, then word
    frequencies from the ``wordfreq`` package: a word is Spanish (English) when its Zipf frequency
    in Spanish (English) exceeds the other by ``margin``; otherwise it is ambiguous.
    """
    lexicon: dict[str, str] = {}
    if lexicon_path:
        lex = pd.read_csv(lexicon_path)
        lexicon = {str(w).lower(): str(l) for w, l in zip(lex.iloc[:, 0], lex.iloc[:, 1])}
    try:
        from wordfreq import zipf_frequency
    except ImportError:
        zipf_frequency = None
    if not lexicon and zipf_frequency is None:
        return None
    cache: dict[str, str] = {}

    def tag(token: str) -> str:
        word = token.lower().strip(".'-")
        if word in cache:
            return cache[word]
        if not word:
            lang = "unk"
        elif word in FILLERS:
            lang = "filler"
        elif any(ch.isdigit() for ch in word):
            lang = "num"
        elif word in lexicon:
            lang = lexicon[word]
        elif _SPANISH_CHARS & set(word):
            lang = "es"
        elif zipf_frequency is None:
            lang = "unk"
        else:
            es, en = zipf_frequency(word, "es"), zipf_frequency(word, "en")
            if es == 0 and en == 0:
                lang = "unk"
            elif es - en >= margin:
                lang = "es"
            elif en - es >= margin:
                lang = "en"
            else:
                lang = "amb"
        cache[word] = lang
        return lang

    return tag


# --------------------------------------------------------------------------- per-system scoring

@dataclass
class SystemScore:
    name: str
    errors: np.ndarray  # per clip S+D+I
    subs: np.ndarray
    dels: np.ndarray
    ins: np.ndarray
    errors_lowercase: np.ndarray  # per clip errors when casing is ignored
    ref_status: list[list[str]]  # per clip, per reference word: "ok" | "sub" | "del"
    hyp_tokens: list[list[str]]
    raw: list[str] = field(default_factory=list)  # un-normalised hypothesis text per clip


def score_system(name: str, hyp_texts: Sequence[str], ref_tokens: Sequence[list[str]],
                 normalize: Callable[[str], str]) -> SystemScore:
    n = len(ref_tokens)
    errors, subs, dels, ins, errors_lc = (np.zeros(n, dtype=np.int64) for _ in range(5))
    statuses, hyp_tokens = [], []
    for i, (text, ref) in enumerate(zip(hyp_texts, ref_tokens)):
        hyp = tokenize(normalize(text))
        s, d, ins_i, pairs = align(ref, hyp)
        status = ["ok"] * len(ref)
        for r_idx, h_idx in pairs:
            if r_idx is not None:
                if h_idx is None:
                    status[r_idx] = "del"
                elif ref[r_idx] != hyp[h_idx]:
                    status[r_idx] = "sub"
        s_lc, d_lc, i_lc, _ = align([w.lower() for w in ref], [w.lower() for w in hyp])
        subs[i], dels[i], ins[i] = s, d, ins_i
        errors[i] = s + d + ins_i
        errors_lc[i] = s_lc + d_lc + i_lc
        statuses.append(status)
        hyp_tokens.append(hyp)
    return SystemScore(name, errors, subs, dels, ins, errors_lc, statuses, hyp_tokens, list(hyp_texts))


def word_class_error(score: SystemScore, ref_classes: Sequence[list[str]], wanted: str) -> tuple[float, int]:
    """Share of reference words of a class that were substituted or deleted (insertions excluded)."""
    total = wrong = 0
    for status, classes in zip(score.ref_status, ref_classes):
        for st, c in zip(status, classes):
            if c == wanted:
                total += 1
                wrong += st != "ok"
    return (wrong / total if total else float("nan")), total


# --------------------------------------------------------------------------- combination

def _pick(votes: dict, default):
    best = max(votes.values())
    tied = [k for k, v in votes.items() if v == best]
    if default in tied:
        return default
    return sorted(tied, key=lambda k: (k is None, str(k)))[0]


def medoid(hyps: Sequence[list[str]], weights: Sequence[float]) -> int:
    """Index of the hypothesis with the lowest weighted edit distance to the others (MBR)."""
    costs = []
    for i, a in enumerate(hyps):
        costs.append(sum(w * Levenshtein.distance(a, b) for j, (b, w) in enumerate(zip(hyps, weights)) if j != i))
    return int(np.argmin(costs))  # ties go to the earlier system


def rover(hyps: Sequence[list[str]], weights: Sequence[float], pivot: int) -> list[str]:
    """Word-level voting over hypotheses aligned to a pivot (ties keep the pivot's choice).

    Each pivot word position is voted on (a word or deletion) and each gap between pivot words
    is voted on (an inserted phrase or nothing).
    """
    base = hyps[pivot]
    n = len(base)
    slot_votes = [defaultdict(float) for _ in range(n)]
    gap_votes = [defaultdict(float) for _ in range(n + 1)]
    for k, (hyp, w) in enumerate(zip(hyps, weights)):
        covered: list[Optional[str]] = [None] * n
        inserted: dict[int, list[str]] = defaultdict(list)
        if k == pivot:
            covered = list(base)
        else:
            last = -1
            _, _, _, pairs = align(base, hyp)
            for p_idx, h_idx in pairs:
                if p_idx is None:
                    inserted[last + 1].append(hyp[h_idx])
                else:
                    covered[p_idx] = hyp[h_idx] if h_idx is not None else None
                    last = p_idx
        for i in range(n):
            slot_votes[i][covered[i]] += w
        for g in range(n + 1):
            gap_votes[g][tuple(inserted.get(g, ()))] += w
    out: list[str] = []
    for g in range(n + 1):
        out.extend(_pick(gap_votes[g], ()))
        if g < n:
            word = _pick(slot_votes[g], base[g])
            if word is not None:
                out.append(word)
    return out


def combine(scores: Sequence[SystemScore], weights: Sequence[float], method: str) -> list[str]:
    """Per clip combined hypothesis text for ``method`` in {"rover", "mbr"}."""
    texts = []
    for clip in range(len(scores[0].hyp_tokens)):
        hyps = [s.hyp_tokens[clip] for s in scores]
        pivot = medoid(hyps, weights)
        if method == "mbr":
            texts.append(scores[pivot].raw[clip])
        else:
            texts.append(" ".join(rover(hyps, weights, pivot)))
    return texts


# --------------------------------------------------------------------------- statistics

def bootstrap(err: np.ndarray, nref: np.ndarray, idx: np.ndarray) -> np.ndarray:
    """Corpus WER for every bootstrap resample (rows of idx index units: clips or groups)."""
    return err[idx].sum(axis=1) / np.maximum(nref[idx].sum(axis=1), 1)


def group_totals(values: np.ndarray, groups: np.ndarray) -> np.ndarray:
    uniq, inv = np.unique(groups, return_inverse=True)
    out = np.zeros(len(uniq), dtype=np.float64)
    np.add.at(out, inv, values)
    return out


def fmt(x: float, digits: int = 4, sign: bool = False) -> str:
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return "–"
    return f"{x:+.{digits}f}" if sign else f"{x:.{digits}f}"


def parse_by(spec: str) -> tuple[str, Optional[list[float]]]:
    """``col`` for categorical breakdowns, ``col:10,30,60`` to bucket a numeric column."""
    if ":" in spec:
        col, edges = spec.split(":", 1)
        return col, [float(e) for e in edges.split(",") if e]
    return spec, None


def bucket(values: pd.Series, edges: list[float]) -> pd.Series:
    labels = []
    bounds = [-np.inf] + edges + [np.inf]
    for lo, hi in zip(bounds[:-1], bounds[1:]):
        if np.isinf(lo):
            labels.append(f"<{hi:g}")
        elif np.isinf(hi):
            labels.append(f">={lo:g}")
        else:
            labels.append(f"{lo:g}-{hi:g}")
    return pd.cut(values.astype(float), bins=bounds, labels=labels, right=False).astype(str)


# --------------------------------------------------------------------------- main

def parse_pred_arg(arg: str) -> tuple[str, Path]:
    if "=" in arg and not Path(arg).exists():
        name, path = arg.split("=", 1)
        return name, Path(path)
    return Path(arg).stem, Path(arg)


def main(argv: Optional[Sequence[str]] = None) -> dict:
    ap = argparse.ArgumentParser(description=__doc__.split("\n\n")[0])
    ap.add_argument("preds", nargs="+", help="prediction CSVs (submission format); NAME=PATH to rename")
    ap.add_argument("--ref", required=True, help="reference CSV with audio_filename,transcript")
    ap.add_argument("--scorer", help="path to the official score.py (default: search for a runtime checkout)")
    ap.add_argument("--baseline", help="system the paired deltas are computed against (default: best WER)")
    ap.add_argument("--bootstrap", type=int, default=10000, help="bootstrap resamples (0 to skip)")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--meta", help="CSV with audio_filename plus metadata columns (speaker, duration_s, ...)")
    ap.add_argument("--cluster-col", help="meta column to resample as a unit (e.g. speaker) instead of clips")
    ap.add_argument("--by", action="append", default=[],
                    help="meta column to break WER down by; COL:10,30,60 buckets a numeric column")
    ap.add_argument("--combine", help="'all' or comma-separated systems to combine with ROVER and MBR")
    ap.add_argument("--weights", help="comma-separated NAME=WEIGHT votes for --combine (default 1)")
    ap.add_argument("--lang-lexicon", help="CSV word,lang (es/en/amb) overriding the wordfreq tagger")
    ap.add_argument("--no-lang", action="store_true", help="skip the Spanish/English breakdown")
    ap.add_argument("--allow-missing", action="store_true", help="score missing clips as empty transcripts")
    ap.add_argument("--out-dir", help="write summary.json, per_clip.csv and combined CSVs here")
    args = ap.parse_args(argv)

    normalize = load_normalizer(find_scorer(args.scorer))
    ref = read_transcripts(Path(args.ref)).sort_index()
    clips = list(ref.index)
    ref_tokens = [tokenize(normalize(t)) for t in ref.values]
    nref = np.array([len(t) for t in ref_tokens], dtype=np.int64)
    if nref.sum() == 0:
        sys.exit("Reference transcripts are empty after normalisation.")

    systems: list[SystemScore] = []
    for arg in args.preds:
        name, path = parse_pred_arg(arg)
        pred = read_transcripts(path)
        missing = [c for c in clips if c not in pred.index]
        if missing and not args.allow_missing:
            sys.exit(f"{path}: missing {len(missing)} of {len(clips)} reference clips (first: {missing[0]}); "
                     "use --allow-missing to score them as empty")
        if missing:
            print(f"warning: {name} has no prediction for {len(missing)} of {len(clips)} clips; "
                  "scored as empty transcripts", file=sys.stderr)
        texts = [pred.get(c, "") for c in clips]
        systems.append(score_system(name, texts, ref_tokens, normalize))
    names = [s.name for s in systems]
    if len(set(names)) != len(names):
        sys.exit(f"Duplicate system names: {names}; use NAME=PATH to disambiguate")
    inputs = list(systems)

    # Combinations are scored like any other system.
    if args.combine:
        chosen = names if args.combine == "all" else [c.strip() for c in args.combine.split(",")]
        unknown = set(chosen) - set(names)
        if unknown:
            sys.exit(f"--combine: unknown systems {sorted(unknown)}")
        weight_map = {}
        for item in (args.weights or "").split(","):
            if item:
                k, v = item.split("=")
                weight_map[k] = float(v)
        members = [s for s in systems if s.name in chosen]
        weights = [weight_map.get(s.name, 1.0) for s in members]
        if len(members) >= 2:
            label = ",".join(s.name for s in members)
            for method in ("rover", "mbr"):
                texts = combine(members, weights, method)
                systems.append(score_system(f"{method.upper()}({label})", texts, ref_tokens, normalize))

    wer = {s.name: s.errors.sum() / nref.sum() for s in systems}
    order = sorted(systems, key=lambda s: wer[s.name])
    baseline = args.baseline or order[0].name
    if baseline not in wer:
        sys.exit(f"--baseline {baseline} is not one of {list(wer)}")

    # Oracle: best single input system per clip (a floor for any selection-based combination).
    oracle = np.min(np.stack([s.errors for s in inputs]), axis=0)

    # Bootstrap over clips, or over clusters (speakers/sessions) when --cluster-col is given.
    meta = None
    if args.meta:
        meta = pd.read_csv(args.meta).drop_duplicates("audio_filename").set_index("audio_filename").reindex(clips)
    units_err = {s.name: s.errors.astype(np.float64) for s in systems}
    units_err["__oracle__"] = oracle.astype(np.float64)
    units_nref = nref.astype(np.float64)
    if args.cluster_col:
        if meta is None or args.cluster_col not in meta.columns:
            sys.exit("--cluster-col needs --meta with that column")
        groups = meta[args.cluster_col].fillna("unknown").astype(str).to_numpy()
        units_err = {k: group_totals(v, groups) for k, v in units_err.items()}
        units_nref = group_totals(units_nref, groups)
    stats: dict[str, dict] = {}
    if args.bootstrap > 0:
        rng = np.random.default_rng(args.seed)
        idx = rng.integers(0, len(units_nref), size=(args.bootstrap, len(units_nref)))
        boot = {k: bootstrap(v, units_nref, idx) for k, v in units_err.items()}
        for s in systems:
            delta = boot[s.name] - boot[baseline]
            stats[s.name] = {
                "ci": np.percentile(boot[s.name], [2.5, 97.5]).tolist(),
                "delta_ci": np.percentile(delta, [2.5, 97.5]).tolist(),
                "p_better": float(np.mean(delta < 0)),
            }
        stats["__oracle__"] = {"ci": np.percentile(boot["__oracle__"], [2.5, 97.5]).tolist()}

    # Diagnostics.
    ref_fill = [["filler" if w.lower().strip(".") in FILLERS else "word" for w in toks] for toks in ref_tokens]
    tagger = None if args.no_lang else make_lang_tagger(args.lang_lexicon)
    ref_lang, ref_switch = None, None
    if tagger:
        ref_lang = [[tagger(w) for w in toks] for toks in ref_tokens]
        ref_switch = []
        for langs in ref_lang:
            prev, marks = None, []
            for lang in langs:
                definite = lang in ("es", "en")
                marks.append("switch" if definite and prev is not None and lang != prev else "no")
                if definite:
                    prev = lang
            ref_switch.append(marks)

    total_ref = int(nref.sum())
    rows = []
    for s in order:
        row = {
            "system": s.name,
            "wer": float(wer[s.name]),
            "sub": float(s.subs.sum() / total_ref),
            "del": float(s.dels.sum() / total_ref),
            "ins": float(s.ins.sum() / total_ref),
            "wer_ignoring_case": float(s.errors_lowercase.sum() / total_ref),
            "filler_error": word_class_error(s, ref_fill, "filler")[0],
        }
        if tagger:
            row["es_error"] = word_class_error(s, ref_lang, "es")[0]
            row["en_error"] = word_class_error(s, ref_lang, "en")[0]
            row["switch_error"] = word_class_error(s, ref_switch, "switch")[0]
        row.update(stats.get(s.name, {}))
        rows.append(row)

    # ---- report
    print(f"Reference: {args.ref} | {len(clips)} clips | {total_ref} words"
          + (f" | bootstrap over {len(units_nref)} {args.cluster_col or 'clip'} units" if stats else ""))
    if tagger:
        counts = defaultdict(int)
        for langs in ref_lang:
            for lang in langs:
                counts[lang] += 1
        n_switch = sum(m == "switch" for marks in ref_switch for m in marks)
        print("Reference words: " + ", ".join(f"{k} {v}" for k, v in sorted(counts.items()))
              + f" | switch points {n_switch}")
    print()
    print(f"| system | WER | 95% CI | Δ vs {baseline} | Δ 95% CI | P(beats {baseline}) |")
    print("|---|---|---|---|---|---|")
    for r in rows:
        ci = r.get("ci")
        dci = r.get("delta_ci")
        print(f"| {r['system']} | {fmt(r['wer'])} | {fmt(ci[0]) + '–' + fmt(ci[1]) if ci else '–'} | "
              f"{fmt(r['wer'] - wer[baseline], sign=True)} | "
              f"{fmt(dci[0], sign=True) + ' to ' + fmt(dci[1], sign=True) if dci and r['system'] != baseline else '–'} | "
              f"{fmt(r['p_better'], 2) if 'p_better' in r and r['system'] != baseline else '–'} |")
    oracle_wer = oracle.sum() / nref.sum()
    oci = stats.get("__oracle__", {}).get("ci")
    print(f"| oracle (best input per clip) | {fmt(oracle_wer)} | "
          f"{fmt(oci[0]) + '–' + fmt(oci[1]) if oci else '–'} | {fmt(oracle_wer - wer[baseline], sign=True)} | – | – |")
    print()
    head = "| system | sub | del | ins | WER ignoring case | filler error |"
    if tagger:
        head += " es-word error | en-word error | switch-word error |"
    print(head)
    print("|" + "---|" * (head.count("|") - 1))
    for r in rows:
        line = (f"| {r['system']} | {fmt(r['sub'])} | {fmt(r['del'])} | {fmt(r['ins'])} | "
                f"{fmt(r['wer_ignoring_case'])} | {fmt(r['filler_error'], 3)} |")
        if tagger:
            line += f" {fmt(r['es_error'], 3)} | {fmt(r['en_error'], 3)} | {fmt(r['switch_error'], 3)} |"
        print(line)

    breakdowns = {}
    for spec in args.by:
        col, edges = parse_by(spec)
        if meta is None or col not in meta.columns:
            sys.exit(f"--by {col}: needs --meta with that column")
        values = bucket(meta[col], edges) if edges else meta[col].fillna("unknown").astype(str)
        levels = sorted(values.unique(), key=str)
        table = {}
        print()
        print("| system | " + " | ".join(f"{col}={lv} (n={int((values == lv).sum())})" for lv in levels) + " |")
        print("|---|" + "---|" * len(levels))
        for s in order:
            cells = []
            for lv in levels:
                mask = (values == lv).to_numpy()
                v = s.errors[mask].sum() / max(nref[mask].sum(), 1)
                table.setdefault(s.name, {})[lv] = float(v)
                cells.append(fmt(v))
            print(f"| {s.name} | " + " | ".join(cells) + " |")
        breakdowns[col] = table

    summary = {
        "reference": str(args.ref),
        "clips": len(clips),
        "reference_words": total_ref,
        "baseline": baseline,
        "systems": rows,
        "oracle_wer": float(oracle_wer),
        "oracle_ci": oci,
        "breakdowns": breakdowns,
    }
    if args.out_dir:
        out = Path(args.out_dir)
        out.mkdir(parents=True, exist_ok=True)
        (out / "summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False))
        per_clip = pd.DataFrame({"audio_filename": clips, "ref_words": nref})
        for s in systems:
            per_clip[f"errors:{s.name}"] = s.errors
        per_clip["errors:oracle"] = oracle
        per_clip.to_csv(out / "per_clip.csv", index=False)
        for s in systems:
            if "(" in s.name:
                fname = re.sub(r"[^A-Za-z0-9_.-]+", "_", s.name).strip("_") + ".csv"
                pd.DataFrame({"audio_filename": clips, "transcript": s.raw}).to_csv(out / fname, index=False)
        print(f"\nWrote {out}/summary.json, per_clip.csv" + (" and combined CSVs" if args.combine else ""))
    return summary


if __name__ == "__main__":
    main()


In [ ]:
%%writefile main.py
"""Lost in Transcription submission: a fine-tuned Whisper (CTranslate2) run with faster-whisper.

Zip this file together with the exported ``model/`` directory (main.py at the zip root). It reads
``submission_format.csv`` and ``clips/`` from the data directory and writes ``submission.csv``.
Logs carry only counts and timings: the rules forbid clip names or other test information in logs.
"""

import os
import time
from pathlib import Path

import pandas as pd
from faster_whisper import BatchedInferencePipeline, WhisperModel

HERE = Path(__file__).resolve().parent
DATA_DIR = Path(os.environ.get("LIT_DATA_DIR", "/code_execution/data"))
SUBMISSION_PATH = Path(os.environ.get("LIT_SUBMISSION_PATH", "/code_execution/submission/submission.csv"))
BEAM_SIZE = int(os.environ.get("LIT_BEAM_SIZE", "5"))
BATCH_SIZE = int(os.environ.get("LIT_BATCH_SIZE", "16"))


def load_model() -> WhisperModel:
    device = os.environ.get("LIT_DEVICE")
    if device is None:
        import ctranslate2

        device = "cuda" if ctranslate2.get_cuda_device_count() > 0 else "cpu"
    return WhisperModel(str(HERE / "model"), device=device, compute_type="float16" if device == "cuda" else "int8")


def main() -> None:
    t0 = time.time()
    submission = pd.read_csv(DATA_DIR / "submission_format.csv")
    pipeline = BatchedInferencePipeline(load_model())
    print(f"model loaded in {time.time() - t0:.0f}s; {len(submission)} clips", flush=True)
    texts, failures = [], 0
    for i, name in enumerate(submission["audio_filename"]):
        try:
            # VAD splits long clips into <=30 s chunks that are decoded as a batch; the language is
            # detected per clip, as in training.
            segments, _ = pipeline.transcribe(str(DATA_DIR / "clips" / name), batch_size=BATCH_SIZE,
                                              beam_size=BEAM_SIZE, vad_filter=True, language=None)
            text = " ".join(s.text.strip() for s in segments).strip()
        except Exception as exc:  # keep going; one bad clip must not cost the whole run
            failures += 1
            print(f"clip {i}: {type(exc).__name__}", flush=True)
            text = ""
        # Empty transcripts are rejected by the platform; "." normalises to nothing when scored.
        texts.append(text or ".")
        if (i + 1) % 50 == 0:
            print(f"{i + 1}/{len(submission)} clips in {time.time() - t0:.0f}s", flush=True)
    submission["transcript"] = texts
    SUBMISSION_PATH.parent.mkdir(parents=True, exist_ok=True)
    submission[["audio_filename", "transcript"]].to_csv(SUBMISSION_PATH, index=False)
    print(f"wrote {len(submission)} rows ({failures} failures) in {time.time() - t0:.0f}s", flush=True)


if __name__ == "__main__":
    main()


## 1. Data

In [ ]:
import json
import shutil
import tarfile
import zipfile

RAW = SCRATCH / "raw"
RAW.mkdir(parents=True, exist_ok=True)


def mdc_download(dataset_id, name):
    import requests
    from kaggle_secrets import UserSecretsClient

    key = UserSecretsClient().get_secret("MDC_API_KEY")
    r = requests.post(f"https://mozilladatacollective.com/api/datasets/{dataset_id}/download",
                      headers={"Authorization": f"Bearer {key}", "Content-Type": "application/json"}, timeout=60)
    r.raise_for_status()
    target = RAW / f"{name}.tar.gz"
    with requests.get(r.json()["downloadUrl"], stream=True, timeout=600) as dl, open(target, "wb") as f:
        dl.raise_for_status()
        for chunk in dl.iter_content(1 << 20):
            f.write(chunk)
    print(f"downloaded {name}: {target.stat().st_size / 1e6:.0f} MB")


def extract_archives(root):
    for arc in sorted(p for p in root.rglob("*") if p.is_file() and p.name.endswith((".tar.gz", ".tgz", ".tar", ".zip"))):
        dest = RAW / arc.name.split(".")[0]
        if dest.exists():
            continue
        print("extracting", arc.name)
        if arc.suffix == ".zip":
            with zipfile.ZipFile(arc) as z:
                z.extractall(dest)
        else:
            with tarfile.open(arc) as t:
                try:
                    t.extractall(dest, filter="data")
                except TypeError:  # Python without extraction filters
                    t.extractall(dest)


for name, dataset_id in MDC_DATASET_IDS.items():
    if dataset_id and not (RAW / f"{name}.tar.gz").exists():
        mdc_download(dataset_id, name)
if INPUT_ROOT.exists():
    extract_archives(INPUT_ROOT)
extract_archives(RAW)

roots = [r for r in (INPUT_ROOT, RAW) if r.exists()]
candidates = [d for r in roots for d in [r, *r.rglob("*")] if d.is_dir()]
MIAMI_DIR = next((d for d in candidates if (d / "audios").is_dir() and any((d / "chat").glob("*.cha"))), None)
DEV_DIR = next((d for d in candidates if (d / "clips").is_dir()
                and ((d / "metadata.tsv").is_file() or (d / "metadata.csv").is_file())), None)
assert MIAMI_DIR, "No Miami data found: expected a folder with chat/*.cha and audios/ (see the setup notes)"
assert DEV_DIR, "No dev set found: expected a folder with metadata.tsv (or .csv) and clips/ (see the setup notes)"
print("Miami:", MIAMI_DIR, f"({len(list((MIAMI_DIR / 'chat').glob('*.cha')))} transcripts)")
print("Dev:  ", DEV_DIR, f"({len(list((DEV_DIR / 'clips').iterdir()))} clips)")


In [ ]:
import urllib.request

SCORER = SCRATCH / "score.py"  # the organisers' scorer, so every WER here matches the leaderboard's
if not SCORER.exists():
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/drivendataorg/lost-in-transcription-runtime/main/score.py", SCORER)
print("scorer:", SCORER)


In [ ]:
PREP = SCRATCH / "prepared"
if not (PREP / "stats.json").exists():
    !python prepare_data.py --miami "{MIAMI_DIR}" --dev "{DEV_DIR}" --out "{PREP}" --workers {os.cpu_count()}
stats = json.loads((PREP / "stats.json").read_text())
print(json.dumps({k: v for k, v in stats.items() if k != "convention_audit"}, indent=2))
print("\n".join(stats["convention_audit"]))

train_sets = [PREP / "miami_train.csv"]
DEV_EVAL_GOLD = {"none": PREP / "gold_dev_all.csv", "half": PREP / "gold_dev_b.csv", "all": None}[DEV_FOR_TRAINING]
if DEV_FOR_TRAINING == "half":
    train_sets.append(PREP / "dev_a.csv")
elif DEV_FOR_TRAINING == "all":
    train_sets.append(PREP / "dev_all.csv")
print("train on:", [p.name for p in train_sets], "| dev score on:", DEV_EVAL_GOLD.name if DEV_EVAL_GOLD else "nothing")


## 2. Zero-shot baseline on the dev clips held out from training

In [ ]:
EVAL_DIR = WORK / "dev_eval"
if RUN_ZERO_SHOT_BASELINE and DEV_EVAL_GOLD is not None:
    # Same decoding as the submission: VAD chunks, batch 16, beam 5, language detected per clip.
    !python run_systems.py --manifest "{DEV_EVAL_GOLD}" --clips "{DEV_DIR / 'clips'}" --out-dir "{EVAL_DIR}" --system "zeroshot=whisper:{ZERO_SHOT_MODEL}?batch=16"


## 3. Fine-tune (LoRA)

In [ ]:
RUN = SCRATCH / "run"
cpu_flag = "" if N_GPUS else "--cpu"
train_arg = ",".join(str(p) for p in train_sets)
!torchrun --standalone --nproc_per_node={max(1, N_GPUS)} train_whisper.py --train "{train_arg}" --eval "{PREP / 'miami_holdout.csv'}" --scorer "{SCORER}" --output "{RUN}" --model {BASE_MODEL} --max-steps {MAX_STEPS} --eval-steps {EVAL_STEPS} --eval-samples {EVAL_SAMPLES} --lr {LR} --batch-size {BATCH_PER_GPU} --effective-batch {EFFECTIVE_BATCH} --lora-r {LORA_R} --lora-alpha {LORA_ALPHA} --encoder-from-layer {ENCODER_FROM_LAYER} --augment-prob {AUGMENT_PROB} --resume {cpu_flag}

summary = json.loads((RUN / "summary.json").read_text())
print("held-out Miami WER by step (step 0 = zero-shot):")
for row in summary["eval_history"]:
    print(f"  step {row['step']:>4}: {row['wer']:.4f}")
print("best:", summary["best_checkpoint"], summary["best_eval_wer"])
shutil.copytree(RUN / "best_adapter", WORK / "best_adapter", dirs_exist_ok=True)
_ = shutil.copy(RUN / "summary.json", WORK / "train_summary.json")


## 4. Export to CTranslate2 and score on dev

In [ ]:
CT2 = SCRATCH / "ct2_model"
!python export_whisper.py --adapter "{RUN / 'best_adapter'}" --out-ct2 "{CT2}" --quantization {CT2_QUANTIZATION}


In [ ]:
if DEV_EVAL_GOLD is not None:
    !python run_systems.py --manifest "{DEV_EVAL_GOLD}" --clips "{DEV_DIR / 'clips'}" --out-dir "{EVAL_DIR}" --overwrite --system "finetuned=whisper:{CT2}?batch=16"
    !python compare.py --ref "{DEV_EVAL_GOLD}" {EVAL_DIR}/*.csv --scorer "{SCORER}" --meta "{EVAL_DIR / 'info' / 'meta.csv'}" --by duration_s:10,30,60 --out-dir "{WORK / 'dev_eval_report'}"
else:
    print("DEV_FOR_TRAINING = 'all': every dev clip was trained on, so there is no dev score.")


## 5. Package and check the submission

In [ ]:
import pandas as pd

SUB = SCRATCH / "submission_src"
shutil.rmtree(SUB, ignore_errors=True)
SUB.mkdir(parents=True)
shutil.copy("main.py", SUB / "main.py")
shutil.copytree(CT2, SUB / "model")
ZIP = WORK / "submission.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_STORED) as z:  # weights barely compress; storing is faster
    for f in sorted(SUB.rglob("*")):
        if f.is_file():
            z.write(f, f.relative_to(SUB))
print(f"{ZIP} ({ZIP.stat().st_size / 1e9:.2f} GB)")

# Run the zip the way the platform does: unpack, run main.py against a copy of the data layout.
CHECK = SCRATCH / "zip_check"
shutil.rmtree(CHECK, ignore_errors=True)
(CHECK / "data").mkdir(parents=True)
with zipfile.ZipFile(ZIP) as z:
    z.extractall(CHECK / "src")
gold = pd.read_csv(DEV_EVAL_GOLD if DEV_EVAL_GOLD is not None else PREP / "gold_dev_all.csv")
gold[["audio_filename"]].assign(transcript="").to_csv(CHECK / "data" / "submission_format.csv", index=False)
(CHECK / "data" / "clips").symlink_to(DEV_DIR / "clips")
!cd "{CHECK / 'src'}" && LIT_DATA_DIR="{CHECK / 'data'}" LIT_SUBMISSION_PATH="{CHECK / 'submission.csv'}" python main.py
if DEV_EVAL_GOLD is not None:
    !python "{SCORER}" "{DEV_EVAL_GOLD}" --predicted-path "{CHECK / 'submission.csv'}"


## Next steps

- **Download** `submission.zip` from this notebook's *Output* and upload it on the competition's
  submission page. `best_adapter/`, `train_summary.json` and `dev_eval_report/` are saved as well.
- **Read the dev report** above. Keep a change only if the paired Δ interval against the
  alternative excludes 0.
- **Final model**: set `DEV_FOR_TRAINING = "all"` and run again.
- **Other checkpoints**: to score one besides the best, point `export_whisper.py --adapter` at
  `/tmp/lit/run/checkpoint-<step>` in the same session, then re-run the evaluation cell.
